# Pipeline Setup

In [23]:
# importing libraries
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.metrics import  r2_score, mean_absolute_error
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [25]:
# load data
df = pd.read_csv('../data/Student Social Media And Mental Health Impact.csv')
# dropping duplicate vlaues
df.drop_duplicates(inplace=True)
# converting negetive physical activity hr values to zero
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)
# Dropping Country, Country of use
df.drop(columns=['Country', 'Purpose_Of_Use'], inplace=True)

X, Y = df.iloc[:,0:10], df.iloc[:,-1]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [26]:
skwewd_col         = ['Study_Hours']
other_numeric_cols = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks","Physical_Activity_Hours","Sleep_Hours_Per_Night"]
ordinal_col        = ['Stress_Level']
normal_col         = ["Gender", "Academic_Level", "Most_Used_Platform"]


X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=149)

In [27]:
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])


plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])


ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))
])


nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skwewd_col),
    ("Plain_Numeric",plain_numeric_pipeline, other_numeric_cols ),
    ('Ordinal', ordinal_pipeline, ordinal_col),
    ('Normal', nominal_pipeline, normal_col)
])

In [28]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor())
])
rf_pipeline.fit(X_train, Y_train)
rf_preds = rf_pipeline.predict(X_test)

print('r2 score on training data : ', r2_score(Y_train, rf_pipeline.predict(X_train)))
print('r2 score on testing data : ', r2_score(Y_test, rf_preds))

print('MAE on testing data : ', mean_absolute_error(Y_test, rf_preds))

r2 score on training data :  0.9807909856111962
r2 score on testing data :  0.8779203554900421
MAE on testing data :  0.34217216666666683


In [29]:
import joblib
joblib.dump(rf_pipeline, '../models/mental_health_score_model.pkl')

['../models/mental_health_score_model.pkl']